In [2]:
# ===============================
# 1 Import Libraries
# ===============================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import flwr as fl  

from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.10.0


In [3]:
# ===============================
# 2 GPU Configuration
# ===============================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPUs:", gpus)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict


def get_patient_id(filename):
    """
    Extract patient ID from filename like:
    OAS1_0001_MR1_mpr-1_100.jpg -> 0001
    """
    return filename.split("_")[1]


def split_dataset_patientwise(
    input_dir, output_dir, split_ratio=(0.7, 0.15, 0.15), seed=42
):
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    train_ratio, val_ratio, test_ratio = split_ratio
    assert round(train_ratio + val_ratio + test_ratio, 5) == 1.0

    classes = ["Mild Dementia", "Non Dementia", "Very mild Dementia"]
    splits = ["train", "val", "test"]

    # Create folder structure
    for split in splits:
        for cls in classes:
            (output_path / split / cls).mkdir(parents=True, exist_ok=True)

    random.seed(seed)

    for cls in classes:
        cls_dir = input_path / cls

        if not cls_dir.exists():
            print(f"Warning: {cls_dir} not found")
            continue

        # 🔥 Group files by patient ID
        patient_dict = defaultdict(list)

        for f in cls_dir.iterdir():
            if f.is_file() and not f.name.startswith("."):
                pid = get_patient_id(f.name)
                patient_dict[pid].append(f)

        patients = list(patient_dict.keys())
        random.shuffle(patients)

        total_patients = len(patients)
        train_idx = int(total_patients * train_ratio)
        val_idx = train_idx + int(total_patients * val_ratio)

        train_patients = patients[:train_idx]
        val_patients = patients[train_idx:val_idx]
        test_patients = patients[val_idx:]

        def copy_patient_files(patient_list, split_name):
            count = 0
            for pid in patient_list:
                for f in patient_dict[pid]:
                    dest = output_path / split_name / cls / f.name
                    shutil.copy2(f, dest)
                    count += 1
            return count

        train_count = copy_patient_files(train_patients, "train")
        val_count = copy_patient_files(val_patients, "val")
        test_count = copy_patient_files(test_patients, "test")

        print(f"\nClass: {cls}")
        print(
            f"Patients -> Train: {len(train_patients)}, Val: {len(val_patients)}, Test: {len(test_patients)}"
        )
        print(f"Images   -> Train: {train_count}, Val: {val_count}, Test: {test_count}")


# --- Run ---
split_dataset_patientwise(
    input_dir="Data", output_dir="Dataset_split", split_ratio=(0.70, 0.15, 0.15)
)


Class: Mild Dementia
Patients -> Train: 14, Val: 3, Test: 4
Images   -> Train: 3294, Val: 732, Test: 976

Class: Non Dementia
Patients -> Train: 186, Val: 39, Test: 41
Images   -> Train: 47092, Val: 9699, Test: 10431

Class: Very mild Dementia
Patients -> Train: 40, Val: 8, Test: 10
Images   -> Train: 9333, Val: 1952, Test: 2440


In [7]:
TRAIN_PATH = r"Dataset_split/train"
VAL_PATH = r"Dataset_split/val"
TEST_PATH = r"Dataset_split/test"

In [8]:
# ===============================
# 3 Paths and Parameters
# ===============================

# Each PC will use its OWN local dataset
TRAIN_DIR = "Dataset_split/train"
VAL_DIR = "Dataset_split/val"
TEST_DIR = "Dataset_split/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

CLASSES = ["Mild Dementia", "Non Dementia", "Very mild Dementia"]

In [9]:
# CLIENT DATA
def load_client_data():
    ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
    )

    ds = ds.map(
        lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.map(
        lambda x, y: (data_augmentation(x, training=True), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    return ds.prefetch(tf.data.AUTOTUNE)


# EVAL DATA
def load_eval_data(path):
    ds = tf.keras.utils.image_dataset_from_directory(
        path, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
    )

    ds = ds.map(
        lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )

    return ds.prefetch(tf.data.AUTOTUNE)

In [10]:
# ===============================
# 6 Test Dataset
# ===============================

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    class_names=CLASSES,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# 🔥 ADD THIS
test_dataset = test_dataset.map(
    lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

Found 13847 files belonging to 3 classes.


In [11]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        epsilon = 1e-7
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)

        loss = weight * cross_entropy
        return tf.reduce_sum(loss, axis=1)

    return loss

In [12]:
def create_model():
    base = tf.keras.applications.DenseNet121(
        include_top=False, weights="imagenet", input_shape=(224, 224, 3)
    )

    # Better for FL
    for layer in base.layers[:-30]:
        layer.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    output = tf.keras.layers.Dense(3, activation="softmax")(x)

    model = tf.keras.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=focal_loss(),
        metrics=["accuracy"],
        run_eagerly=False,
    )

    return model

In [13]:
def load_client_data():
    ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
    )

    # DenseNet preprocessing
    ds = ds.map(
        lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )

    # Augmentation (train only)
    ds = ds.map(
        lambda x, y: (data_augmentation(x, training=True), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    return ds.prefetch(tf.data.AUTOTUNE)

In [ ]:

#this is the server code
import flwr as fl

fl.server.start_server(
    server_address="0.0.0.0:8080",
    config=fl.server.ServerConfig(num_rounds=5),
    strategy=fl.server.strategy.FedAvg(),
)

INFO flwr 2026-05-01 20:44:37,300 | app.py:148 | Starting Flower server, config: ServerConfig(num_rounds=5, round_timeout=None)
INFO flwr 2026-05-01 20:44:37,344 | app.py:168 | Flower ECE: gRPC server running (5 rounds), SSL is disabled
INFO flwr 2026-05-01 20:44:37,345 | server.py:86 | Initializing global parameters
INFO flwr 2026-05-01 20:44:37,345 | server.py:273 | Requesting initial parameters from one random client


In [ ]:
#this is the client code
import flwr as fl
import tensorflow as tf

# ===============================
# CONFIG
# ===============================
TRAIN_DIR = "Dataset_split/train"
VAL_DIR = "Dataset_split/val"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 2
ROUNDS = 5

class_weights = {0: 5.0, 1: 0.4, 2: 2.0}

# ===============================
# DATA LOADING
# ===============================
from tensorflow.keras.applications.densenet import preprocess_input


def load_client_data():
    ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
    )
    ds = ds.map(lambda x, y: (preprocess_input(x), y))
    return ds.prefetch(tf.data.AUTOTUNE)


def load_val_data():
    ds = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
    )
    ds = ds.map(lambda x, y: (preprocess_input(x), y))
    return ds.prefetch(tf.data.AUTOTUNE)


# ===============================
# MODEL
# ===============================
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        epsilon = 1e-7
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)
        return tf.reduce_sum(weight * cross_entropy, axis=1)

    return loss


def create_model():
    base = tf.keras.applications.DenseNet121(
        include_top=False, weights="imagenet", input_shape=(224, 224, 3)
    )

    for layer in base.layers[:-30]:
        layer.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    output = tf.keras.layers.Dense(3, activation="softmax")(x)

    model = tf.keras.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=focal_loss(),
        metrics=["accuracy"],
        run_eagerly=False,
    )

    return model


# ===============================
# FLOWER CLIENT
# ===============================
class FlowerClient(fl.client.NumPyClient):
    def __init__(self):
        self.model = create_model()
        self.train_data = load_client_data()
        self.val_data = load_val_data()

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        self.model.fit(
            self.train_data, epochs=EPOCHS, class_weight=class_weights, verbose=1
        )

        # 🔥 Save final global model
        if config["server_round"] == ROUNDS:
            self.model.save("final_global_model.h5")
            print("✅ Final Global Model Saved!")

        return self.model.get_weights(), len(self.train_data), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)

        loss, acc = self.model.evaluate(self.val_data, verbose=0)
        return loss, len(self.val_data), {"accuracy": acc}


# ===============================
# START CLIENT
# ===============================
fl.client.start_numpy_client(server_address="172.16.104.164:8080", client=FlowerClient())

Found 59719 files belonging to 3 classes.


INFO flwr 2026-05-01 20:42:06,088 | grpc.py:50 | Opened insecure gRPC connection (no certificates were passed)
DEBUG flwr 2026-05-01 20:42:06,091 | connection.py:39 | ChannelConnectivity.IDLE
DEBUG flwr 2026-05-01 20:42:06,095 | connection.py:39 | ChannelConnectivity.CONNECTING


Found 12383 files belonging to 3 classes.


DEBUG flwr 2026-05-01 20:42:08,133 | connection.py:39 | ChannelConnectivity.TRANSIENT_FAILURE
DEBUG flwr 2026-05-01 20:42:08,349 | connection.py:113 | gRPC channel closed


_MultiThreadedRendezvous: <_MultiThreadedRendezvous of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNAVAILABLE: ipv4:172.16.104.164:8080: ConnectEx: Connection refused (No connection could be made because the target machine actively refused it.
 -- 10061)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"failed to connect to all addresses; last error: UNAVAILABLE: ipv4:172.16.104.164:8080: ConnectEx: Connection refused (No connection could be made because the target machine actively refused it.\r\n -- 10061)", grpc_status:14}"
>